# A2.3 — Les matrices pour les premiers modèles

B (MLP) et C (plus proches voisins) doivent pouvoir modéliser sans attendre le chargement de tout le train
(sprint 3). On leur fournit deux sous-ensembles, construits par `python -m casmi.matrices`
(code : `src/casmi/matrices.py`) :

| Sous-ensemble | Spectres | Filtre | Rôle |
|---|---|---|---|
| `dev` | ceux des 2 000 molécules `dev` | `query_ok` | évaluer (requêtes de validation) |
| `train_sample` | 200 000 tirés au hasard dans le split `train` | `keep` | entraîner |

Pour chacun, trois sortes de fichiers dans `data/processed/`, alignés ligne à ligne :
- `X_{nom}.npz` : les spectres binés (matrice creuse, 10 000 colonnes, lignes de norme 1) ;
- `meta_{nom}_onehot.parquet` et `meta_{nom}_decompose.parquet` : les features de métadonnées, pour les deux
  encodages de l'adduct ;
- `rows_{nom}.parquet` : le `row_id` et l'`inchikey14` (la réponse) de chaque ligne.

Les spectres et les métadonnées sont séparés, car les deux usages diffèrent : le MLP prend les deux, la similarité
cosinus ne doit comparer que les spectres. `load_matrices` les assemble à la demande.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from casmi.config import DATA_PROCESSED
from casmi.matrices import load_encoder, load_matrices
from casmi.split import load_split

pd.set_option("display.width", 200)

## 1. Ce que contiennent les fichiers

In [ ]:
rows = []
for name in ["dev", "train_sample"]:
    X, r, names = load_matrices(name, meta=False)
    files = [f for f in DATA_PROCESSED.iterdir() if f.name.startswith((f"X_{name}.", f"rows_{name}.", f"meta_{name}_"))]
    norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
    rows.append({"sous-ensemble": name, "spectres": X.shape[0], "molécules": r["inchikey14"].nunique(),
                 "cases non nulles par spectre": round(X.nnz / X.shape[0], 1), "spectres vides": int((norms == 0).sum()),
                 "taille sur disque (Mo)": round(sum(f.stat().st_size for f in files) / 1e6, 1),
                 "taille en mémoire (Mo)": round((X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6, 1)})
pd.DataFrame(rows).set_index("sous-ensemble")

Les « spectres vides » n'ont aucun pic sous 1 000 m/z (`MAX_MZ`) : leur vecteur est nul. Ils sont gardés, pour que
chaque requête de `dev` ait une ligne, mais un modèle ne pourra s'appuyer que sur leurs métadonnées.

## 2. Vérifications

In [ ]:
_, rows_dev, _ = load_matrices("dev", meta=False)
_, rows_train, _ = load_matrices("train_sample", meta=False)
split = load_split().set_index("inchikey14")
print("Molécules communes à dev et train_sample :", len(set(rows_dev["inchikey14"]) & set(rows_train["inchikey14"])))
print("Toutes les lignes de dev sont des molécules dev :", split.loc[rows_dev["inchikey14"], "dev"].all())
print("Toutes les lignes de train_sample sont du split train :", (split.loc[rows_train["inchikey14"], "split"] == "train").all())

Les fichiers de métadonnées contiennent aussi le `row_id` : `load_matrices` vérifie qu'il est dans le même ordre que
`rows_{nom}.parquet`, et s'arrête sinon.

## 3. Comment s'en servir

**Pour le MLP (B)** : spectres et métadonnées, avec l'encodage de l'adduct de son choix.

In [ ]:
X, rows, names = load_matrices("train_sample", adduct_encoding="onehot")
print(X.shape, "| dernières colonnes :", names[-3:])
X, rows, names = load_matrices("train_sample", adduct_encoding="decompose")
print(X.shape, "| colonnes d'adduct :", [n for n in names if n.startswith("adduct")])

**Pour les plus proches voisins (C)** : les spectres seuls. Leurs lignes sont de norme 1, donc le produit de
matrices donne directement la similarité cosinus. À faire par blocs de requêtes, en ne gardant que les meilleurs
voisins de chaque bloc :

In [ ]:
X_dev, rows_dev, _ = load_matrices("dev", meta=False)
X_train, rows_train, _ = load_matrices("train_sample", meta=False)
sim = (X_dev[:1000] @ X_train.T).toarray()  # 1 000 requêtes × 200 000 spectres
print(f"Similarité cosinus du plus proche voisin (1 000 requêtes) : médiane {np.median(sim.max(axis=1)):.2f}")

Les molécules de `dev` ne sont jamais dans le train : le plus proche voisin est donc toujours une autre molécule.

**Pour encoder d'autres spectres** (toute la validation, le test) : l'encodeur de métadonnées appris sur le split
`train` est sauvegardé.

In [ ]:
encoder = load_encoder("onehot")
print(len(encoder.adducts), "adducts gardés ;", "imputeur d'énergie :", type(encoder.ce_model).__name__)

## 4. Et pour tout le train ?

Avec environ 97 cases non nulles par spectre, tout le split `train` (2,26 millions de spectres `keep`) tiendrait en
environ 1,8 Go en matrice creuse, contre 90 Go en dense. C'est l'objet du sprint 3 (`make_loader`) : charger par
morceaux plutôt que tout d'un coup.